In [ ]:
import os
import json

import chromadb
import nest_asyncio
nest_asyncio.apply()  # 이벤트 루프 중복 방지(jupyter 환경인 경우)

from dotenv import load_dotenv
load_dotenv()

from llama_index.core.schema import TextNode
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.google_genai import GoogleGenAI

# 환경 설정(Google API Key)

In [ ]:
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

# 모델 설정(Embedding, LLM)

In [ ]:
def setup_model():
    # 모델 설정
    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")
    llm = GoogleGenAI(model="gemini-3-flash-preview",
                      api_key=GOOGLE_API_KEY)

    # LlamaIndex 설정
    Settings.llm = llm
    Settings.embed_model = embed_model

    print("임베딩, LLM 모델 설정 완료!")

In [ ]:
setup_model()

# Node 불러오기

In [ ]:
with open("./nodes.json", 'r', encoding='utf-8') as f:
    node_dicts = json.load(f)

nodes = [TextNode.from_dict(d) for d in node_dicts]

# 인덱싱 및 Vector DB 저장

In [ ]:
def setup_db(nodes):
    # paper_id 추출
    paper_id = nodes[0].metadata.get("paper_id")
    if paper_id is None:
        raise ValueError("nodes metadata에 paper_id가 없습니다.")
    
    # ChromaDB
    db = chromadb.PersistentClient(path="./chromadb")
    chroma_collection = db.get_or_create_collection("papaer_collection")

    # Vector Store, Storage Context 생성
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    existing = chroma_collection.get(where={"paper_id": paper_id},
                                     include=[])
    existing_ids = set(existing["ids"])

    # 인덱스 생성 및 저장(같은 논문이면 로드, 아니면 생성)
    if len(existing["ids"]) > 0:
        print(f"이미 인덱싱된 논문(paper_id={paper_id}) → 기존 인덱스 로드.")
        index = VectorStoreIndex.from_vector_store(vector_store, storage_context=storage_context)
    else:
        print("새로운 인덱스 생성...", end='')
        index = VectorStoreIndex(nodes, storage_context=storage_context)
        print("저장 완료!")

    return index

In [ ]:
def retrieval(index, query):
    print("\n--- [검색 테스트] ---\n")

    # 검색
    retriever = index.as_retriever(similarity_top_k=1000)
    retrieved_nodes = retriever.retrieve(query)

    # for i, node in enumerate(retrieved_nodes):
    #     print(f"[검색 결과 {i+1}] (Score: {node.score:.4f})")
    #     print(f"내용 요약: {node.node.get_content()[:150]}...")

    # LLM 답변 생성
    print("\n---Gemini 답변 생성 ---\n")
    query_engine = index.as_query_engine()
    response = query_engine.query(query)
    print(f"최종 답변: {response}")

In [ ]:
index = setup_db(nodes)

In [ ]:
query = "abstract 섹션에 대해서 설명해줘"
retrieval(index, query)